# Vectora — run 3 (300 papers, the new metrics)

Run 3 measures what the milestones ask for, on a corpus big enough that recall is not a ceiling.

| # | Experiment | Milestone |
|---|---|---|
| A | Retrieval, all modes: **passage-level recall** vs paper-level | evidence-level recall, a few hundred papers |
| B | Refusal calibration on the new corpus | refusal accuracy |
| C | Answers in `fast` mode, two answer models: **token F1**, **hallucinated-citation rate** | answer quality, pre-strip citations |
| D | Answers in `deep` mode (grounding judge on) | judged faithfulness |
| E | Ablation, re-run now the Qdrant race is fixed | run 2's ablation was invalid |
| F | Export 30 judged answers for hand labelling | judge agreement |

Answer model: `gemma3:4b` (run 2's best). Judge and utility model: `qwen2.5:7b`, a *different* model, so the judge never grades its own answers.

**Kaggle setup:** right panel → Session options → Accelerator **GPU T4 x2** (or P100), **Internet on** (needs a phone-verified account).
Then **Save Version → Save & Run All (Commit)**. That runs in the background: you can close the tab and the laptop. Results appear under the version's **Output** tab (`results/`).
Uses roughly 4–5 h of your weekly Kaggle GPU quota. Expect roughly 3–5 hours in total; D is the long one.

## 1. Output folder

In [ ]:
import datetime, os
RUN = "run3_" + datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
OUT = f"/kaggle/working/results/{RUN}"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

## 2. Ollama on the GPU

In [ ]:
ANSWER = "gemma3:4b"     # run 2's best answer model
JUDGE  = "qwen2.5:7b"    # utility + judge: a different model, so it does not grade itself
MODELS = [ANSWER, JUDGE]

# pciutils + lshw let the installer detect the GPU; without them it installs a CPU-only build.
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT,
                          env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1", "OLLAMA_NUM_PARALLEL": "4",
                               "OLLAMA_MAX_LOADED_MODELS": "2", "OLLAMA_CONTEXT_LENGTH": "8192"})
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2); print("ollama up"); break
    except Exception:
        time.sleep(1)
for m in MODELS:
    !ollama pull {m} 2>&1 | tail -1
!ollama run {ANSWER} "hi" > /dev/null 2>&1; ollama ps
!grep -i -E "inference compute|no compatible gpu" /kaggle/working/ollama.log | head -3

## 3. Vectora + config

In [ ]:
!git clone -q https://github.com/SiddharthaRaja/Vectora.git /kaggle/working/Vectora
%cd /kaggle/working/Vectora
!git log --oneline -1 | tee {OUT}/commit.txt
!pip install -q -e ".[ingest,eval]" 2>&1 | grep -v -i "warning\|dependency resolver\|incompatible" | tail -2
open(".env", "w").write(f"""
OPENAI_API_KEY=ollama
OPENAI_BASE_URL=http://127.0.0.1:11434/v1
VECTORA_ANSWER_MODEL={ANSWER}
VECTORA_UTILITY_MODEL={JUDGE}
VECTORA_DAILY_COST_CEILING_USD=0
VECTORA_LLM_TIMEOUT_SECONDS=180
VECTORA_LLM_MAX_RETRIES=2
VECTORA_CORPUS=qasper-300
VECTORA_QDRANT_URL=
VECTORA_QDRANT_PATH=./data/qdrant
VECTORA_MIN_DENSE_SCORE=0.40
VECTORA_MIN_RERANK_SCORE=-4.0
VECTORA_CORS_ORIGINS=http://localhost:5000
VECTORA_LOG_LEVEL=WARNING
""")

## 4. Build the 300-paper corpus, ingest, index

Same command as on your laptop, so the same 300 papers and 365 title-free questions (the golden set is also in git; `--force` rebuilds it identically).
Every case has the annotators' evidence paragraphs and all their answers.

In [ ]:
!python scripts/qasper_corpus.py --corpus qasper-300 --split all --papers 300 --force 2>&1 | tail -6
!git diff --stat corpora/qasper-300/golden.yaml | tail -1 | tee {OUT}/golden_diff.txt   # empty = identical to the committed set
!date; vectora ingest 2>&1 | tail -3
!vectora index --recreate 2>&1 | tail -3; date
!cp corpora/qasper-300/golden.yaml {OUT}/golden.yaml

## 5. Experiment A: retrieval, all three modes

`recall` is now **passage-level**: a chunk only counts if it contains a paragraph the annotators marked as evidence. `paper` is the old paper-level number, shown next to it.
The gap between them is how often Vectora finds the right paper but the wrong part of it.

In [ ]:
!date
!vectora eval --output {OUT}/expA_retrieval.json 2>&1 | tee {OUT}/expA_retrieval.log | tail -30

## 6. Experiment B: refusal calibration on the new corpus

In [ ]:
!vectora calibrate --mode fast 2>&1 | tee {OUT}/expB_calibrate_fast.log | tail -16
!vectora calibrate 2>&1 | tee {OUT}/expB_calibrate_balanced.log | tail -22

## 7. Experiment C: answer quality in `fast` mode, two answer models

New in the report: **raw citation validity / hallucinated markers** (what the model wrote, before Vectora strips bad citations) and **answer token F1 / token recall** against QASPER's gold answers.
Mechanical scores only (`--no-judge`), so this is quick.

In [ ]:
!date
for m in [ANSWER, JUDGE]:
    tag = m.replace(":", "-")
    print("=====", m)
    !VECTORA_ANSWER_MODEL={m} vectora answers --mode fast --no-judge --output {OUT}/expC_fast_{tag}.json 2>&1 | tee {OUT}/expC_fast_{tag}.log | grep -A22 -i "mechanical" | head -24

## 8. Experiment D: `deep` mode with the faithfulness judge

Deep mode runs the grounding check (judged by `qwen2.5:7b`) and the relevance judge. This is the long one, roughly 1–2 hours for 365 questions + 8 negatives.

In [ ]:
!date
!vectora answers --mode deep --output {OUT}/expD_deep.json 2>&1 | tee {OUT}/expD_deep.log | tail -45
!date

## 9. Experiment E: ablation, re-run

Run 2's ablation was invalid: the Qdrant race corrupted vectors when searches ran in parallel. That is fixed and tested now, so this is the first valid ablation.
Check the baseline row matches experiment A's balanced recall; if it does, the race is really gone.

In [ ]:
!date
!vectora ablate --output {OUT}/expE_ablate.json 2>&1 | tee {OUT}/expE_ablate.log | tail -40

## 10. Experiment F: export 30 answers for hand labelling

Writes `judge_labels.csv` to your Drive folder. Later, on your laptop: open it, **hide the `judge_*` columns**, and for each row put `1` in `human_faithful` if every claim in the answer is supported by the sources shown, `0` if any claim is not.
Then send it to me and I'll compute agreement.

In [ ]:
!python -m evaluation.judge_agreement export --answers {OUT}/expD_deep.json --chunks corpora/qasper-300/chunks.jsonl --out {OUT}/judge_labels.csv --n 30

## 11. Wrap up

In [ ]:
!cp corpora/qasper-300/reports/* {OUT}/ 2>/dev/null; cp /kaggle/working/ollama.log {OUT}/ 2>/dev/null; date | tee {OUT}/finished_at.txt
!ls -la {OUT}
!rm -rf /kaggle/working/Vectora/data /kaggle/working/Vectora/corpora/qasper-300/papers   # keep the Output tab small